# 03 — The ReAct loop

Module 02 was one round, sometimes two, that we typed by hand. An agent is that round **until it is done**.

In module 02 the official loop was silent: the model decided, you executed, you fed the result back. The reasoning happened. You never saw it. Only `tool_calls` surfaced.

**ReAct** (Reason + Act) is the pattern that makes the model narrate that reasoning as plain text, interleaved with actions:

1. **Thought** — what it is trying to figure out next, written out loud.
2. **Action** — one tool, in a fixed text format.
3. **PAUSE** — it stops and waits, on purpose.
4. **Observation** — you run the action and feed the result back as text.

That is the idea of this module: **thinking out loud, plus the ability to act**. The thought is not decoration. It is how you debug, how you trust, and how you see the plan change after an observation.

Three parts today. They are not alternatives.

1. **ReAct, built in the open** — one sequential question, every piece visible, then the `for`.
2. **The same ReAct, live** — a new question, thoughts still on the page, and a running cost. We are not wrapping it in a class; a framework will do that later.
3. **The official loop** — `tool_calls`, no Thought on the page. This is what you will ship. It is not ReAct.


## 1. Learn

```mermaid
flowchart LR
    T["Thought"] --> A["Action"]
    A --> C["your code runs a tool"]
    C --> O["Observation"]
    O --> T
    T -->|"writes finish, or we hit the turn cap"| E["Final answer"]
```

Where this sits on the course:

```
00  the API object
01  what an agent is, and when it should not be one
02  the model does not call the tool
03  you are here — think out loud, then act; then the silent loop you keep
04  the same loop, coding tools, then break it
05  the list gets expensive
06  MCP — tools that travel
07  async — two tool calls at once
08  the OpenAI Agents SDK
09  LangGraph — you draw the arrows
10  charts and a sandbox
11-12  retrieval, then agentic RAG
13  delegation
14  security — attack the agent from 12
15  evals and cost
16  the platform landscape
17  process re-engineering
```

ReAct predates `tool_calls`. It works on any chat model that can follow a format. Native function calling is more reliable because the API enforces the shape. ReAct is more transparent because the Thought is on the page. Most frameworks today use `tool_calls` under the hood and still borrow ReAct's idea — think before you act — in how they prompt.

| | ReAct (this part) | Official loop (the last part) |
|---|---|---|
| How the model asks for a tool | A text line: `Action:` | `finish_reason == tool_calls` |
| How you detect it | Walk the lines | A field on the response |
| Reasoning visible? | Yes, `Thought:` | No, inside the model |
| Breaks if the model rambles? | Yes | No, the API holds the shape |
| Works on any chat model? | Yes | Only models with a tool-calling API |

Thoughts are **output tokens**. A chatty plan is a bill, the same `usage` object from module 00. We will print that on the second example.

Choosing a model for this loop is not choosing the smartest one. Reliability and cost compound: a model that is right 95% of the time fails roughly one loop in three at six turns. Pin `reasoning_effort="none"` so a short lookup does not spend silent tokens thinking — `"low"` plus tools returns a 400 on this family. Route the easy turns; do not standardise on the expensive model. Module 15 is where we put a dollar cap next to the turn cap.

Three things that go wrong the first time:

1. No stop condition. Always cap the turns.
2. Dropping the list. `messages` is the memory. Observations stay on it.
3. Raising when a tool fails. Send the error back as text. The next thought needs it.

The first question cannot be answered in one look-up. The second tool needs the result of the first.


## 2. Do

### Load the environment and the two files

Same files as module 02. Same two functions. The notebook stands alone.


In [1]:
from pathlib import Path
import csv
import json
import os

from dotenv import load_dotenv, find_dotenv
from openai import OpenAI


load_dotenv(find_dotenv(usecwd=True))
ROOT = Path(find_dotenv(usecwd=True)).parent

api_key = os.environ.get("OPENAI_API_KEY", "").strip()
model = os.environ.get("MODEL_DEFAULT", "").strip()
assert api_key, "OPENAI_API_KEY is missing. Copy .env.example to .env and add the class key."
assert model, "MODEL_DEFAULT is missing from .env."
price_in = float(os.environ["PRICE_INPUT_PER_MILLION"])
price_out = float(os.environ["PRICE_OUTPUT_PER_MILLION"])

client = OpenAI()
facts_rows = list(csv.DictReader(open(ROOT / "data" / "fun_facts.csv")))
flight_rows = list(csv.DictReader(open(ROOT / "data" / "flight_data.csv")))
print("OPENAI_API_KEY is set:", True)
print("MODEL_DEFAULT:", model)


OPENAI_API_KEY is set: True
MODEL_DEFAULT: gpt-5.4-nano


In [2]:
def get_fact(city):
    needle = city.strip().lower()
    for row in facts_rows:
        if row["City"].lower() == needle:
            return row["Fun Fact"]
    return "no fact for that city"


def get_flight(from_city, to_city):
    found = []
    a = from_city.strip().lower()
    b = to_city.strip().lower()
    for row in flight_rows:
        if row["from_city"].lower() == a and row["to_city"].lower() == b:
            found.append(row["price"] + " dollars, " + row["duration"] + " minutes")
    if not found:
        return "no flight found"
    return "; ".join(found)


print("Barcelona -> Dubai: ", get_flight("Barcelona", "Dubai"))
print("Barcelona -> Amman: ", get_flight("Barcelona", "Amman"))
print("Dubai fact:         ", get_fact("Dubai"))


Barcelona -> Dubai:  646.86 dollars, 829 minutes
Barcelona -> Amman:  909.99 dollars, 404 minutes
Dubai fact:          Dubai is home to the tallest building in the world, the Burj Khalifa.


Those three prints are the true answer. Dubai is cheaper (646.86 vs 909.99). The model cannot see them unless we send them.

### The question

Not "a flight and a fact" in parallel, and not a landing city already written in the prompt. The fact is about **whichever city is cheaper**. The model does not know that city until both flights return.


In [3]:
QUESTION = "From Barcelona, is it cheaper to fly to Dubai or to Amman? Give me a fun fact about whichever one is cheaper."
print(QUESTION)


From Barcelona, is it cheaper to fly to Dubai or to Amman? Give me a fun fact about whichever one is cheaper.


### Part 1 — ReAct, one thought, stopped

No `tools=` argument. The available actions live in the system prompt. The model writes three lines and stops. We have not parsed anything yet.

Read the prompt, then the reply, with your eye. Find `Action` and `Action Input`.


In [4]:
SYSTEM_TEXT = """You help with flights and city facts. You cannot see the data.

Reply with exactly these lines and then stop:
Thought: <why you are doing this>
Action: <get_flight or get_fact or finish>
Action Input: <for get_flight: City1, City2 / for get_fact: City / for finish: the answer>

Do not invent Observation. Do not guess prices or facts.
After an Observation, think again.

Example:
Thought: I need both prices before I can pick the cheaper city.
Action: get_flight
Action Input: Paris, Toronto
"""

print(SYSTEM_TEXT)


You help with flights and city facts. You cannot see the data.

Reply with exactly these lines and then stop:
Thought: <why you are doing this>
Action: <get_flight or get_fact or finish>
Action Input: <for get_flight: City1, City2 / for get_fact: City / for finish: the answer>

Do not invent Observation. Do not guess prices or facts.
After an Observation, think again.

Example:
Thought: I need both prices before I can pick the cheaper city.
Action: get_flight
Action Input: Paris, Toronto



In [5]:
messages = [
    {"role": "system", "content": SYSTEM_TEXT},
    {"role": "user", "content": QUESTION},
]

response = client.chat.completions.create(
    model=model,
    messages=messages,
    max_completion_tokens=160,
    reasoning_effort="none",
)
text = response.choices[0].message.content
print("finish_reason:", response.choices[0].finish_reason)
print(text)


finish_reason: stop
Thought: I need flight prices from Barcelona to Dubai and to Amman to determine which is cheaper, then I can fetch a fun fact about the cheaper destination.
Action: get_flight
Action Input: Barcelona, Dubai


### Pull the action out of the text

No regex. Walk the lines. We will reuse this in the loop, so it gets a name.


In [6]:
def parse_action(text):
    action = None
    action_input = ""
    for line in (text or "").splitlines():
        line = line.strip()
        lower = line.lower()
        if lower.startswith("action input:"):
            action_input = line.split(":", 1)[1].strip()
        elif lower.startswith("action:"):
            action = line.split(":", 1)[1].strip()
    return action, action_input


action, action_input = parse_action(text)
print("action:      ", action)
print("action_input:", action_input)


action:       get_flight
action_input: Barcelona, Dubai


### Run that one action

Same rule as module 02: our code decides. If the name is unknown, we do not raise. We return a sentence the next thought can read.


In [7]:
def run_text_action(action, action_input):
    if action is None:
        return "no Action line found; use get_flight, get_fact, or finish"
    name = action.lower()
    if name == "finish":
        return None
    if name == "get_fact":
        return get_fact(action_input)
    if name == "get_flight":
        parts = [p.strip() for p in action_input.split(",")]
        if len(parts) != 2:
            return "get_flight needs two cities, written City1, City2"
        return get_flight(parts[0], parts[1])
    return "unknown action; use get_flight, get_fact, or finish"


observation = run_text_action(action, action_input)
print("observation:", observation)


observation: 646.86 dollars, 829 minutes


### Give the observation back, one more thought

`messages` is still the memory. We append the assistant text, then a user turn that is only the observation. Then we call again. No loop yet.


In [8]:
messages.append({"role": "assistant", "content": text})
messages.append({"role": "user", "content": "Observation: " + str(observation)})

response = client.chat.completions.create(
    model=model,
    messages=messages,
    max_completion_tokens=160,
    reasoning_effort="none",
)
text2 = response.choices[0].message.content
print(text2)
print()
print("parsed:", parse_action(text2))


Thought: I have the Barcelona→Dubai option; I still need Barcelona→Amman prices to compare.
Action: get_flight
Action Input: Barcelona, Amman

parsed: ('get_flight', 'Barcelona, Amman')


The second thought should now ask for the other flight, or name the cheaper city and ask `get_fact`. That is the middle of ReAct. The observation changed the plan. You can read why.

We are not going to keep copying `create`. Every piece above is one line in a loop: call, parse, maybe stop, run, append.

### The ReAct loop, assembled

Same question, fresh list. Cap of 6. Print every thought and every observation. If `Action` is `finish`, stop. If parse fails, send that sentence back — do not crash.


In [9]:
messages = [
    {"role": "system", "content": SYSTEM_TEXT},
    {"role": "user", "content": QUESTION},
]

for turn in range(6):
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        max_completion_tokens=160,
        reasoning_effort="none",
    )
    text = response.choices[0].message.content
    print("--- turn", turn + 1, "---")
    print(text)
    messages.append({"role": "assistant", "content": text})

    action, action_input = parse_action(text)
    print("parsed:", action, "|", action_input)

    if action is not None and action.lower() == "finish":
        print("stop: finish")
        break

    observation = run_text_action(action, action_input)
    print("observation:", observation)
    messages.append({"role": "user", "content": "Observation: " + str(observation)})


--- turn 1 ---
Thought: I need both prices before I can determine which destination is cheaper from Barcelona.
Action: get_flight
Action Input: Barcelona, Dubai
parsed: get_flight | Barcelona, Dubai
observation: 646.86 dollars, 829 minutes


--- turn 2 ---
Thought: I have the Barcelona→Dubai fare; I still need the Barcelona→Amman fare to compare.
Action: get_flight
Action Input: Barcelona, Amman
parsed: get_flight | Barcelona, Amman
observation: 909.99 dollars, 404 minutes


--- turn 3 ---
Thought: Dubai is cheaper than Amman based on the fares I retrieved, so I can provide the answer and a fun fact about Dubai.
Action: get_fact
Action Input: Dubai
parsed: get_fact | Dubai
observation: Dubai is home to the tallest building in the world, the Burj Khalifa.


--- turn 4 ---
Thought: I can now answer which route is cheaper and include the requested fun fact about the cheaper destination.
Action: finish
Action Input: Dubai is cheaper than Amman to fly from Barcelona (646.86 USD vs 909.99 USD). Fun fact: Dubai is home to the tallest building in the world, the Burj Khalifa.
parsed: finish | Dubai is cheaper than Amman to fly from Barcelona (646.86 USD vs 909.99 USD). Fun fact: Dubai is home to the tallest building in the world, the Burj Khalifa.
stop: finish


That `for` is the ReAct agent. The Thought lines are the point. You just paid for every one of them as completion tokens.

We have been holding `messages` in the open. That is the right way to learn. The next example is the same `for`, a live question, and a running cost — so you can see that a chatty plan is a bill. Module 09 will hide this list inside an object. The loop does not change.


### Part 2 — Same ReAct, a live question

Module 00 asked Prague for the weather and the model invented a number. Today we actually fetch it. [Open-Meteo](https://open-meteo.com/) is a free forecast API. No key. No signup.

The question needs two lookups and a subtraction:

> Is Prague warmer than Amsterdam right now, and by how many degrees?

Two tools: `get_weather` (city to a temperature) and `subtract` (two numbers). No `eval`. Test them with no model first.


In [10]:
import requests


def get_weather(city):
    try:
        geo = requests.get(
            "https://geocoding-api.open-meteo.com/v1/search",
            params={"name": city, "count": 1},
            timeout=10,
        ).json()
    except requests.RequestException as exc:
        return "weather lookup failed: " + str(exc)
    results = geo.get("results") or []
    if not results:
        return "no such city"
    lat = results[0]["latitude"]
    lon = results[0]["longitude"]
    try:
        wx = requests.get(
            "https://api.open-meteo.com/v1/forecast",
            params={"latitude": lat, "longitude": lon, "current": "temperature_2m"},
            timeout=10,
        ).json()
    except requests.RequestException as exc:
        return "weather lookup failed: " + str(exc)
    temp = wx["current"]["temperature_2m"]
    return str(temp) + " C"


def subtract(a, b):
    return str(float(a) - float(b))


print("Prague:    ", get_weather("Prague"))
print("Amsterdam:", get_weather("Amsterdam"))
print("15 - 10:   ", subtract("15", "10"))


Prague:     17.4 C


Amsterdam: 19.2 C
15 - 10:    5.0


If those first two lines failed, the VM cannot reach Open-Meteo. Skip the rest of Part 2. The ReAct idea already landed in Part 1.

The action runner is the same shape as `run_text_action`. The `for` below is the one you already wrote, pointed at these two tools.


In [11]:
def run_weather_action(action, action_input):
    if action is None:
        return "no Action line found; use get_weather, subtract, or finish"
    name = action.lower()
    if name == "finish":
        return None
    if name == "get_weather":
        return get_weather(action_input)
    if name == "subtract":
        parts = [p.strip() for p in action_input.split(",")]
        if len(parts) != 2:
            return "subtract needs two numbers, written a, b"
        try:
            return subtract(parts[0], parts[1])
        except ValueError:
            return "subtract needs two numbers"
    return "unknown action; use get_weather, subtract, or finish"


Same loop as Part 1. New tools. Watch the Thoughts. Watch the bill move. We are not wrapping it in a class — a framework will do that later. The running cost is the point.


In [12]:
SYSTEM_WEATHER = """You compare live weather. You cannot see the temperatures until you look them up.

Reply with exactly these lines and then stop:
Thought: <why>
Action: <get_weather or subtract or finish>
Action Input: <for get_weather: City / for subtract: a, b / for finish: the answer>

Do not invent Observation. Do not guess temperatures.
After an Observation, think again.
"""


Construct the list once. Ask. Print every thought and the running dollar total.


In [13]:
messages = [
    {"role": "system", "content": SYSTEM_WEATHER},
    {
        "role": "user",
        "content": "Is Prague warmer than Amsterdam right now, and by how many degrees?",
    },
]
thought_tokens = 0
cost = 0.0
last_text = ""
for turn in range(6):
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        max_completion_tokens=160,
        reasoning_effort="none",
    )
    usage = response.usage
    thought_tokens = thought_tokens + usage.completion_tokens
    turn_cost = (
        usage.prompt_tokens / 1_000_000 * price_in
        + usage.completion_tokens / 1_000_000 * price_out
    )
    cost = cost + turn_cost
    last_text = response.choices[0].message.content
    print("--- turn", turn + 1, "---")
    print(last_text)
    print(
        "completion_tokens this thought:",
        usage.completion_tokens,
        f"  running cost ${cost:.6f}",
    )
    messages.append({"role": "assistant", "content": last_text})
    action, action_input = parse_action(last_text)
    if action is not None and action.lower() == "finish":
        print("stop: finish")
        break
    observation = run_weather_action(action, action_input)
    print("observation:", observation)
    messages.append({"role": "user", "content": "Observation: " + str(observation)})

print()
print("thought tokens (all turns):", thought_tokens)
print(f"what those thoughts cost:   ${cost:.6f}")


--- turn 1 ---
Thought: I need live weather for both cities, since I can’t know temperatures without looking them up.
Action: get_weather
Action Input: Prague / Amsterdam
completion_tokens this thought: 35   running cost $0.000065


observation: no such city


--- turn 2 ---
Thought: The city name was rejected, so I need the correct city/location entry to look up weather. I should retry using the proper country-qualified city for Prague and Amsterdam.
Action: get_weather
Action Input: Prague, Czechia
completion_tokens this thought: 50   running cost $0.000157


observation: 17.4 C


--- turn 3 ---
Thought: I have Prague’s current temperature (17.4 C), but I still need Amsterdam’s temperature to compare and compute the degree difference.
Action: get_weather
Action Input: Amsterdam
completion_tokens this thought: 41   running cost $0.000251


observation: 19.2 C


--- turn 4 ---
Thought: I have both current temperatures (Prague 17.4 C, Amsterdam 19.2 C). Compute the difference.
Action: subtract
Action Input: 17.4, 19.2
completion_tokens this thought: 46   running cost $0.000363
observation: -1.8000000000000007


--- turn 5 ---
Thought: -1.8 C means Amsterdam is warmer than Prague by 1.8 degrees (since Prague - Amsterdam = -1.8).  
Action: finish
Action Input: Prague is colder than Amsterdam by about 1.8 degrees right now.
completion_tokens this thought: 56   running cost $0.000500
stop: finish

thought tokens (all turns): 228
what those thoughts cost:   $0.000500


Every `Thought:` line was billed as output. A longer plan is not free. Later, when a framework says "reasoning," it is still this: tokens on the way out.

You can still see `messages`. That is the product a wrapper will hide. The loop inside is the one you wrote in Part 1.


### Part 3 — The official loop

This is not a second ReAct. ReAct was the visible Thought. Here the API takes over `Action` and the thought goes back inside the model. You keep the loop. You lose the narration.

That is the trade: more reliable shape, less you can read. Module 04 and module 09 wrap **this** loop, not the text format.

Same Barcelona comparison. Named schemas, then one list, the way we combined tools in module 02.


In [14]:
get_fact_json = {
    "name": "get_fact",
    "description": "Look up a fun fact about a city.",
    "parameters": {
        "type": "object",
        "properties": {"city": {"type": "string"}},
        "required": ["city"],
    },
}

get_flight_json = {
    "name": "get_flight",
    "description": "Look up flights between two cities. Returns price in dollars and duration in minutes.",
    "parameters": {
        "type": "object",
        "properties": {
            "from_city": {"type": "string"},
            "to_city": {"type": "string"},
        },
        "required": ["from_city", "to_city"],
    },
}

tools = [
    {"type": "function", "function": get_fact_json},
    {"type": "function", "function": get_flight_json},
]
print([t["function"]["name"] for t in tools])


['get_fact', 'get_flight']


One `create` with `tools=`. Stop. Look at `finish_reason`. Do not run anything yet.


In [15]:
messages = [{"role": "user", "content": QUESTION}]

response = client.chat.completions.create(
    model=model,
    messages=messages,
    tools=tools,
    max_completion_tokens=160,
    reasoning_effort="none",
)
message = response.choices[0].message
print("finish_reason:", response.choices[0].finish_reason)
print("content:      ", message.content)
print("n tool_calls: ", len(message.tool_calls or []))
if message.tool_calls:
    for call in message.tool_calls:
        print(" asked:", call.function.name, call.function.arguments)


finish_reason: tool_calls
content:       None
n tool_calls:  2
 asked: get_flight {"from_city": "Barcelona", "to_city": "Dubai"}
 asked: get_flight {"from_city": "Barcelona", "to_city": "Amman"}


`finish_reason` should be `tool_calls`. If it asked for `get_fact` without a city from a flight, that is a bad plan — we will still run it and send back `no fact` or a wrong city, and the next thought can recover.

Run whatever it asked. Same `if` as module 02. Append `message`, then each `tool` result.


In [16]:
def run_official_call(call):
    args = json.loads(call.function.arguments)
    if call.function.name == "get_fact":
        return get_fact(args.get("city", ""))
    if call.function.name == "get_flight":
        return get_flight(args.get("from_city", ""), args.get("to_city", ""))
    return "unknown tool"


messages.append(message)
for call in message.tool_calls or []:
    result = run_official_call(call)
    print(call.function.name, "->", result)
    messages.append({"role": "tool", "tool_call_id": call.id, "content": result})


get_flight -> 646.86 dollars, 829 minutes
get_flight -> 909.99 dollars, 404 minutes


One more `create`. If `finish_reason` is `stop`, we have a sentence. If it is `tool_calls` again, we would run the `for` again. That repeat is the loop.


In [17]:
response = client.chat.completions.create(
    model=model,
    messages=messages,
    tools=tools,
    max_completion_tokens=160,
    reasoning_effort="none",
)
message = response.choices[0].message
print("finish_reason:", response.choices[0].finish_reason)
print("content:      ", message.content)
print("n tool_calls: ", len(message.tool_calls or []))


finish_reason: tool_calls
content:       None
n tool_calls:  1


### The official loop, assembled

Fresh list. Same question. Cap of 6. The only new idea: stop when `finish_reason` is not `tool_calls`.


In [18]:
messages = [{"role": "user", "content": QUESTION}]

for turn in range(6):
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        tools=tools,
        max_completion_tokens=160,
        reasoning_effort="none",
    )
    message = response.choices[0].message
    print("--- turn", turn + 1, "finish_reason:", response.choices[0].finish_reason, "---")

    if not message.tool_calls:
        print(message.content)
        break

    messages.append(message)
    for call in message.tool_calls:
        result = run_official_call(call)
        print(call.function.name, "->", result)
        messages.append({"role": "tool", "tool_call_id": call.id, "content": result})


--- turn 1 finish_reason: tool_calls ---
get_flight -> 646.86 dollars, 829 minutes
get_flight -> 909.99 dollars, 404 minutes


--- turn 2 finish_reason: tool_calls ---
get_fact -> Dubai is home to the tallest building in the world, the Burj Khalifa.


--- turn 3 finish_reason: stop ---
From **Barcelona**, it’s **cheaper to fly to Dubai** (**$646.86**) than to **Amman** (**$909.99**).

**Fun fact (Dubai):** Dubai is home to the **tallest building in the world, the Burj Khalifa**.


Same bones as the ReAct `for`: call, maybe stop, run, append. The API took over `Action` / `Action Input`. The thought is now inside the model. You see it less. That is the trade, not an upgrade of ReAct.

This official loop is the artifact later modules will point at. Module 04 puts coding tools behind it. Module 05 watches the list get expensive. Module 09 asks what a framework buys you instead of these twenty lines — a wrapper around this silent loop.


### Part 4 — The same job, OpenAI Agents SDK

Parts 1–3 were the truth: a Thought you can read, then an official `for` you wrote.

`Runner` **is** that `for`. `@function_tool` **is** the schema you typed in Part 3. `Agent` is the system prompt plus the tool list. Same question: cheaper of Barcelona → Dubai vs Barcelona → Amman, then a fact about the winner. Sequential on purpose. The fact city is not known until both flights return.

Use `await Runner.run`. Jupyter already has an event loop; `run_sync` crashes here (you saw that in 02). Module 07 is why the SDK is async. Module 08 is handoffs, sessions, guardrails.

`with trace("03 Barcelona cheaper")` sends a replay to [platform.openai.com/traces](https://platform.openai.com/traces). Each tool call is a span. You should see **three** spans: two flights, then a fact. `draw_graph` draws this agent and its two tools (green ellipses). `model=` is still our OpenAI string. We stay on the class key.


In [19]:
from agents import Agent, Runner, function_tool, trace, gen_trace_id


@function_tool
def fact_tool(city: str) -> str:
    """Look up a fun fact about a city."""
    return get_fact(city)


@function_tool
def flight_tool(from_city: str, to_city: str) -> str:
    """Look up a flight between two cities. Price in dollars, duration in minutes."""
    return get_flight(from_city, to_city)


travel = Agent(
    name="Travel",
    instructions=(
        "Use the tools. Do not invent prices or facts. "
        "Compare both flights before you pick a city, then use fact_tool on the cheaper one."
    ),
    model=model,
    tools=[fact_tool, flight_tool],
)

trace_id = gen_trace_id()
print("Trace:", "https://platform.openai.com/traces/trace?trace_id=" + trace_id)

# Jupyter already has a loop. Use await. Do not call Runner.run_sync.
with trace("03 Barcelona cheaper", trace_id=trace_id):
    sdk_result = await Runner.run(travel, QUESTION)
print(sdk_result.final_output)


Trace: https://platform.openai.com/traces/trace?trace_id=trace_4f9fc65e5ac94a74bb812ff759e9b4af


It’s cheaper to fly from **Barcelona to Dubai** (about **$646.86** for **829 min**) than to **Amman** (about **$909.99** for **404 min**).

**Fun fact about Dubai:** It’s home to the **tallest building in the world**, the **Burj Khalifa**.


### Now look at the trace

If the sentence names Amman and the hills, the SDK used the observation.

Open the printed URL (or [platform.openai.com/traces](https://platform.openai.com/traces)). Two tool spans, in order. That is the official loop you wrote, drawn as a timeline.

Now the map of the agent itself:


In [20]:
try:
    from agents.extensions.visualization import draw_graph

    draw_graph(travel)
except Exception as exc:
    print("Graph not drawn:", exc)
    print("The run still happened. Use the trace link. Install the graphviz program (dot) for the picture.")


Yellow rectangle = the agent. Green ellipses = `fact_tool` and `flight_tool`. You did not write that diagram. The SDK knows the wiring because you passed `tools=[...]`.

You still own `get_fact` and `get_flight`. The model still does not open the CSV. The SDK is a host for the loop you already understand.


## 3. Observe

Print the official `messages` list from Part 3. Count how many times we actually ran a tool. That number is the reason this question needed a loop.

Keep the traces tab from Part 4 open. Same two calls, two views: a Python list, and a timeline.


In [21]:
n_tools = 0
for turn in messages:
    if isinstance(turn, dict):
        role = turn["role"]
        extra = turn.get("content", "") or ""
        if role == "tool":
            n_tools += 1
            extra = extra[:80]
        elif role == "user":
            extra = extra[:80]
        else:
            extra = extra[:80]
    else:
        role = turn.role
        extra = "tool_calls" if turn.tool_calls else (turn.content or "")[:80]
    print(f"  {role:10} {extra}")

print()
print("tool results on the list:", n_tools)


  user       From Barcelona, is it cheaper to fly to Dubai or to Amman? Give me a fun fact ab
  assistant  tool_calls
  tool       646.86 dollars, 829 minutes
  tool       909.99 dollars, 404 minutes
  assistant  tool_calls
  tool       Dubai is home to the tallest building in the world, the Burj Khalifa.

tool results on the list: 3


If `n_tools` is 0, the model guessed Dubai or Amman. That is a chatbot with a good memory, not this loop.

If `n_tools` is 1, it looked up one flight (or jumped to a fact) and skipped the comparison.

If `n_tools` is 2, it compared the flights and invented the fact, or looked up one flight and the fact.

If `n_tools` is 3, the observation did its job: Barcelona → Dubai, Barcelona → Amman, then the Dubai fact.

The file says Barcelona → Dubai is 646.86 dollars / 829 minutes, Barcelona → Amman is 909.99 / 404. Dubai wins. Dubai fact: home to the tallest building in the world, the Burj Khalifa. Compare that to whatever sentence you got.


### When the tool has nothing

Every loop so far succeeded. That is not what a Tuesday looks like.

`get_fact` and `get_flight` do not raise when they miss. They return `"no fact for that city"` and `"no flight found"`. That was a decision back in Part 1, and it is the whole reason the next cell is uneventful: a miss arrives as an **observation**, in the same slot as a hit, and the model reasons about it.

Ask for a city that is in neither CSV.

In [ ]:
messages = [
    {
        "role": "user",
        "content": (
            "Give me a fun fact about Reykjavik, and tell me what a flight "
            "from Barcelona to Reykjavik costs."
        ),
    }
]

for turn in range(6):
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        tools=tools,
        max_completion_tokens=160,
        reasoning_effort="none",
    )
    message = response.choices[0].message
    print("--- turn", turn + 1, "finish_reason:", response.choices[0].finish_reason, "---")

    if not message.tool_calls:
        print(message.content)
        break

    messages.append(message)
    for call in message.tool_calls:
        result = run_official_call(call)
        print(call.function.name, "->", result)
        messages.append({"role": "tool", "tool_call_id": call.id, "content": result})

It said it does not have the data. It did not invent a fact about Reykjavik, and it did not crash.

That is worth naming, because it is the difference between a demo and something you would run:

- **A tool that raises** kills the loop. Your `for` never reaches turn 2.
- **A tool that returns a string** hands the model something it can act on. `no flight found` is a perfectly good observation.
- **A tool that guesses** is the dangerous one. If `get_fact` returned a plausible sentence for a city it had never heard of, nothing downstream could tell it from a real one.

The same three choices apply to the JSON coming the other way. `json.loads` on a malformed `arguments` string raises, and in this notebook that ends the cell. In production it is a `try` that returns `"could not read those arguments"` and lets the next turn correct itself. Module 15 puts a budget cap next to the turn cap for the same reason: the loop needs a way to survive a bad turn, and a way to stop.

## 4. Challenge

Use the **official** loop (the one with `tools=`). A new comparison, same shape:

> From Tokyo, is it cheaper to fly to Moscow or to Berlin? Give me a fun fact about whichever one is cheaper.

Cap the turns. Print each tool result. Bind:

- `n_lookups` — how many tool results you put on the list
- `final_text` — the last `content` when `finish_reason` is not `tool_calls`

The next cell only checks that you looped at least once and that a final sentence exists. It does not score the wording. We will look at a solution in the debrief.


In [ ]:
# n_lookups, final_text = ...


In [ ]:
assert n_lookups >= 1, "the loop should have run at least one tool"
assert final_text and str(final_text).strip(), "final_text should be the last model sentence"
print("n_lookups:", n_lookups)
print(final_text)
